# 🌍 Modelo Predictivo y Auditoría de Fraudes por Barrios
Sistema de triaje con Validación Temporal, SHAP y exportación a CSV. Versión Definitiva 4.0 (Con Módulo de Turismo Oficial).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import scipy.stats as stats
import shap
from scipy.optimize import curve_fit
from sklearn.ensemble import RandomForestRegressor
import warnings
warnings.filterwarnings('ignore')

print("✅ Librerías cargadas correctamente.")

# ==========================================
# 1. CARGA DE DATOS Y ESTANDARIZACIÓN
# ==========================================
print("\n1. Cargando datos...")

archivo_agua = 'AMAEM-2022-2024.csv' 
archivo_clima = 'clima_barrios_alicante_final.csv'
archivo_ndvi = 'ndvi_alicante.csv'
archivo_turismo = 'provincia-info-vt.csv'

try:
    # Cargar Agua, Clima y Satélite
    df_agua = pd.read_csv(archivo_agua)
    df_clima = pd.read_csv(archivo_clima)
    df_ndvi = pd.read_csv(archivo_ndvi)
    
    # Cargar Turismo (Ojo al separador por punto y coma)
    df_turismo = pd.read_csv(archivo_turismo, sep=';')
    
    # --- LIMPIEZA DE AGUA ---
    df_agua = df_agua.rename(columns={'Barrio': 'barrio', 'Fecha (aaaa/mm/dd)': 'fecha_mes', 'Consumo (litros)': 'consumo'})
    df_agua['consumo'] = df_agua['consumo'].astype(str).str.replace(',', '').astype(float)
    df_agua['fecha_mes'] = pd.to_datetime(df_agua['fecha_mes']).dt.strftime('%Y-%m')
    df_agua = df_agua.groupby(['barrio', 'fecha_mes'])['consumo'].sum().reset_index()

    # --- LIMPIEZA DE CLIMA ---
    df_clima = df_clima.rename(columns={'zona': 'barrio', 'fecha': 'fecha_mes'})

    # --- LIMPIEZA DE TURISMO ---
    # Transformar '2024M12' en '2024-12'
    df_turismo['fecha_mes'] = df_turismo['Fecha'].str.replace('M', '-')
    # Usamos 'Número de noches ocupadas' como métrica de turismo, quitando los puntos de miles
    df_turismo['turismo'] = df_turismo['Número de noches ocupadas'].astype(str).str.replace('.', '', regex=False).astype(float)
    df_turismo = df_turismo[['fecha_mes', 'turismo']] # Nos quedamos solo con lo que importa

    # --- FUSIÓN MAESTRA (MERGE) ---
    df_final = pd.merge(df_agua, df_clima, on=['fecha_mes', 'barrio'], how='left')
    df_final = pd.merge(df_final, df_ndvi, on='fecha_mes', how='left')
    df_final = pd.merge(df_final, df_turismo, on='fecha_mes', how='left')

    df_final = df_final.sort_values(['barrio', 'fecha_mes']).reset_index(drop=True)
    
    # Rellenar huecos vacíos para que la IA no falle
    columnas_a_rellenar = df_final.columns.drop('barrio')
    df_final[columnas_a_rellenar] = df_final.groupby('barrio')[columnas_a_rellenar].ffill().bfill()
    df_final = df_final.fillna(df_final.mean(numeric_only=True))

    print(f"✅ Datos cargados y unificados: {len(df_final)} filas. Turismo integrado con éxito.")
except Exception as e:
    print(f"❌ Error al cargar datos: {e}")
    raise

In [ ]:
# ==========================================
# 2. MODELO FOURIER (Tendencia Base)
# ==========================================
print("\n2. Entrenando Modelo Fourier...")

def modelo_fourier(t, m, c, a1, b1, a2, b2):
    w = 2 * np.pi / 12
    return (m * t + c) + (a1*np.cos(w*t) + b1*np.sin(w*t)) + (a2*np.cos(2*w*t) + b2*np.sin(2*w*t))

df_final['prediccion_fourier'] = 0.0

for barrio in df_final['barrio'].unique():
    mask = df_final['barrio'] == barrio
    y = df_final.loc[mask, 'consumo'].values
    
    fechas = pd.to_datetime(df_final.loc[mask, 'fecha_mes'] + '-01')
    t = fechas.map(pd.Timestamp.toordinal).values

    lim_inf = np.percentile(y, 5)
    lim_sup = np.percentile(y, 95)
    mask_clean = (y >= lim_inf) & (y <= lim_sup)

    try:
        coef, _ = curve_fit(modelo_fourier, t[mask_clean], y[mask_clean], p0=[0, np.mean(y), 1000, 1000, 100, 100], maxfev=10000)
        df_final.loc[mask, 'prediccion_fourier'] = modelo_fourier(t, *coef)
    except Exception as e:
        df_final.loc[mask, 'prediccion_fourier'] = np.mean(y)

print("✅ Fourier completado")

In [ ]:
# ==========================================
# 3. MACHINE LEARNING (Random Forest)
# ==========================================
print("\n3. Entrenando Modelo ML...")

df_final['residuo'] = df_final['consumo'] - df_final['prediccion_fourier']
df_ml = pd.get_dummies(df_final, columns=['barrio'])

exogenas = ['tm_mes', 'p_mes', 'ndvi_satelite', 'turismo', 'tm_max', 'hr']
exogenas_reales = [col for col in exogenas if col in df_final.columns]
columnas_barrios = [col for col in df_ml.columns if col.startswith('barrio_')]

X = df_ml[exogenas_reales + columnas_barrios]
y = df_ml['residuo']

ml_model = RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=5, random_state=42, n_jobs=-1)
ml_model.fit(X, y)

df_final['impacto_exogeno'] = ml_model.predict(X)
df_final['prediccion_hibrida'] = df_final['prediccion_fourier'] + df_final['impacto_exogeno']

print("✅ ML completado")

In [ ]:
# ==========================================
# 4. SHAP + TRIAJE (IA Explicable)
# ==========================================
print("\n4. Calculando IA Explicable (SHAP)...")

df_final['error_final'] = df_final['consumo'] - df_final['prediccion_hibrida']
df_final['z_error_final'] = df_final.groupby('barrio')['error_final'].transform(lambda x: stats.zscore(x, ddof=1)).fillna(0)

explainer = shap.TreeExplainer(ml_model)
shap_values = explainer.shap_values(X)
feature_names = X.columns

for col in exogenas_reales:
    df_final[f'litros_culpa_{col}'] = 0

for i, col in enumerate(feature_names):
    if col in exogenas_reales:
        df_final[f'litros_culpa_{col}'] = np.abs(shap_values[:, i])

df_final['litros_culpa_Desconocida'] = np.abs(df_final['error_final'])

columnas_culpa = [f'litros_culpa_{c}' for c in exogenas_reales] + ['litros_culpa_Desconocida']
df_final['total_variacion_litros'] = df_final[columnas_culpa].sum(axis=1).replace(0, 1)

# --- DICCIONARIO PARA NOMBRES LIMPIOS EN EL CSV ---
mapeo_nombres = {
    'tm_mes': 'Temperatura',
    'p_mes': 'Precipitacion',
    'ndvi_satelite': 'Vegetacion',
    'turismo': 'Turismo',
    'tm_max': 'Temperatura_Max',
    'hr': 'Humedad'
}

columnas_porcentajes = []
for col in exogenas_reales:
    nombre_bonito = mapeo_nombres.get(col, col)
    nueva = f'%_{nombre_bonito}'
    df_final[nueva] = (df_final[f'litros_culpa_{col}'] / df_final['total_variacion_litros']) * 100
    columnas_porcentajes.append(nueva)

df_final['%_Causas_Desconocidas'] = (df_final['litros_culpa_Desconocida'] / df_final['total_variacion_litros']) * 100
columnas_porcentajes.append('%_Causas_Desconocidas')

df_final['consumo'] = df_final['consumo'].round(0)
df_final['prediccion_hibrida'] = df_final['prediccion_hibrida'].round(0)
df_final['z_error_final'] = df_final['z_error_final'].round(2)

print("✅ SHAP completado")

In [ ]:
# ==========================================
# 5. EXPORTACIÓN A CSV (Con Ordenación)
# ==========================================
print("\n5. Generando Informes...")

columnas_vista = ['fecha_mes', 'barrio', 'consumo', 'prediccion_hibrida', 'z_error_final'] + columnas_porcentajes
z_leve, z_mod, z_grave = 1.5, 2.0, 2.5

exc_grave = df_final[df_final['z_error_final'] > z_grave][columnas_vista].sort_values('z_error_final', ascending=False)
exc_mod = df_final[(df_final['z_error_final'] > z_mod) & (df_final['z_error_final'] <= z_grave)][columnas_vista].sort_values('z_error_final', ascending=False)
exc_leve = df_final[(df_final['z_error_final'] > z_leve) & (df_final['z_error_final'] <= z_mod)][columnas_vista].sort_values('z_error_final', ascending=False)

def_grave = df_final[df_final['z_error_final'] < -z_grave][columnas_vista].sort_values('z_error_final', ascending=True)
def_mod = df_final[(df_final['z_error_final'] < -z_mod) & (df_final['z_error_final'] >= -z_grave)][columnas_vista].sort_values('z_error_final', ascending=True)
def_leve = df_final[(df_final['z_error_final'] < -z_leve) & (df_final['z_error_final'] >= -z_mod)][columnas_vista].sort_values('z_error_final', ascending=True)

tablas = [exc_grave, exc_mod, exc_leve, def_grave, def_mod, def_leve]
nombres = [
    '1_EXCESO_Grave.csv', '2_EXCESO_Moderado.csv', '3_EXCESO_Leve.csv',
    '4_DEFECTO_Grave.csv', '5_DEFECTO_Moderado.csv', '6_DEFECTO_Leve.csv'
]

for i, tabla in enumerate(tablas):
    tabla = tabla.copy()
    for col in columnas_porcentajes:
        tabla[col] = tabla[col].round(1).astype(str) + '%'
    tabla.to_csv(nombres[i], index=False)

print("✅ 6 Archivos CSV de alertas generados correctamente y ordenados.")

In [ ]:
# ==========================================
# 6. VISUALIZACIÓN
# ==========================================
barrio_ejemplo = df_final['barrio'].unique()[0]
df_plot = df_final[df_final['barrio'] == barrio_ejemplo]

error_std = df_plot['error_final'].std()

plt.figure(figsize=(14,6))

plt.fill_between(
    df_plot['fecha_mes'],
    df_plot['prediccion_hibrida'] - error_std*1.5,
    df_plot['prediccion_hibrida'] + error_std*1.5,
    color='green', alpha=0.1, label='Normalidad (Z < 1.5)'
)

plt.plot(df_plot['fecha_mes'], df_plot['prediccion_hibrida'], 'k--', linewidth=2, label='Predicción IA')
plt.plot(df_plot['fecha_mes'], df_plot['consumo'], 'k-', linewidth=2, label='Consumo Real')

def pintar_alerta(df, mascara, color, label):
    if not df[mascara].empty:
        plt.scatter(df[mascara]['fecha_mes'], df[mascara]['consumo'], c=color, s=100, label=label, zorder=5)

pintar_alerta(df_plot, df_plot['z_error_final'] > 2.5, 'red', '🔴 Grave')
pintar_alerta(df_plot, (df_plot['z_error_final'] > 1.5) & (df_plot['z_error_final'] <= 2.5), 'orange', '🟠 Leve/Mod')
pintar_alerta(df_plot, df_plot['z_error_final'] < -1.5, 'blue', '🔵 Defecto')

plt.xticks(df_plot['fecha_mes'][::3], rotation=45)
plt.title(f'Auditoría y Anomalías: {barrio_ejemplo}', fontsize=16, fontweight='bold')
plt.ylabel('Consumo (Litros)')
plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()